### Install new Libraries

In [13]:
# !pip install ddgs trafilatura -q

### Setup

In [14]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS # duck duck go
import trafilatura
from IPython.display import Markdown, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [15]:
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2)

In [ ]:
def search_web(query: str):
    print("Query:", repr(query))
    try:
        results = DDGS().text(query, max_results=3)
        print("Results:", len(results))
        return json.dumps(results, indent=2)
    except Exception as error:
        print(type(error).__name__, repr(error))
        return f"No search results available for query: {query}"

In [16]:
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f" \u274C Failed to fetch or extract text.")
    return f"Could not extract text from {url}. Try a different source."

### Step 2: Describe as LLM tools

In [17]:
tools = []

In [18]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Return 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function": search_web_function})

In [19]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the content of a URL using trafilatura.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

### Step 3: Tool Call Handler

In [20]:
def handle_tool_call(tool_calls):
    tool_call_results = []
    content = ""

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling function {function_name} with argument: {args}") # for future debugging

        # Route to the appropriate function based on the function name
        if function_name == "search_web":
            result = search_web(args["query"])
            content = f"Search results: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        # elif function_name == "insert_function_name_3":
        #     content = insert_function_name_3(args["message"])
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }
        tool_call_results.append(tool_call_result)

    return tool_call_results

### Step 4: System Prompt

This tells the LLM *who it is* and *how to behave*. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce

In [21]:
RESEARCH_AGENT_PROMPT = """
You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
It CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

### Step 5: The Agentic Loop

In [22]:
def run_research_agent(topic: str, max_iterations: int=10) -> str:
    """
    Run the research agent on a topic and return the research brief.

    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops
    
    Return:
    The research brief as a string
    """
    print(f"\n\U0001F50D Starting research on: {topic}")
    print("=" * 60)
    # Initialize conversation messages list with system prompt + research task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and rpoduce a comprehensive brief:\n\n{topic}"}
    ]

    # LOOP
    iteration = 1
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        # 1. Call the LLM and get response
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message
        messages.append(message)

        # 2. Check if LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        # 3. Otherwise: no tools were called, read message content
        else: 
            content = message.content # No tool calls, so there is just a normal text response from the LLM
            # Check if DONE, then return
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research complete!")
                return research_brief
            
            # Otherwise: not yet done, append message
            else:
                print(f"  \U0001F4AD Agent is thinking:")
                pprint(content)
                # Loop continues to next iteration

        # 4. If we're entering the final iteration, force a final answer
        if (iteration == max_iterations -1):
            print("  \u26a0 Safety limit reached. Stopping research in next iteration.")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your research brief now. You MUST respond with DONE: followed by your brief."})

    # Fallback return
    return "Research incomplete. Maximum iterations reache without finalizing brief."

### Let's run it!

In [23]:
# brief = run_research_agent("AI in Healthcare in 2030")
# display(Markdown(brief))

### 3 types of evals

1. Human Evals
2. Code-based Evals
3. LLM-as-a-judge Evals

### Evals

In [24]:
JUDGE_PROMPT = """
# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research agent **transcript/trace and its deliverable research brief** to determine if there was a **source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Source:** A distinct, externally identifiable origin of information that the agent referenced in the deliverable — e.g., a named publication, website/domain, report, paper, dataset, company page, or interview subject. A source counts whether it appears as a URL, a hyperlink, a bracketed citation, an inline attribution ("according to the WHO 2024 report..."), or a named mention in the body text. Formal citation formatting is **not** required.

**Counting rules:**
- Count **unique** sources only. The same source referenced multiple times counts once.
- Two distinct documents from the same publisher/domain count as **one** source (e.g., two Reuters articles = 1 source).
- The source must be **identifiable** — a reader could locate it. Vague appeals ("studies show," "experts say," "recent research," "industry data") count as **zero**.
- Sources mentioned only in the agent's intermediate reasoning, tool calls, or search results but **not referenced in the deliverable brief** do **not** count.
- Count sources as presented; do not verify whether they exist, are accurate, or actually support the claim.

**Source breadth failure (label TRUE):** Any of the below occurred:

1. **Fewer than 6 unique sources:** The deliverable brief references 5 or fewer distinct identifiable sources.
2. **Padded source count:** The brief appears to reach 6+ only by repeating the same source under different labels, splitting one document into multiple entries, or listing multiple pages from the same domain — after deduplication, fewer than 6 remain.
3. **Unattributed body:** The brief contains 6+ items in a reference list, but the substantive claims in the body draw on 5 or fewer distinct sources, with the remainder listed but never used.
4. **No deliverable produced:** The agent did not produce a research brief, or produced one with no identifiable sources at all.

**No source breadth failure (label FALSE):** The deliverable brief references **6 or more** unique, identifiable sources after applying the counting rules above.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.
"""

In [25]:
TOPICS = [
    "The current state of solid-state battery commercialization",
    "How GLP-1 receptor agonists are reshaping obesity treatment guidelines",
    "Economic effects of remote work on mid-sized U.S. downtowns",
    "The 2015 Paris Agreement's Article 6 carbon market rules and their implementation status",
    "Recent advances in room-temperature superconductor research and replication attempts",
    "The regulatory landscape for autonomous trucking in the European Union",
    "How streaming economics changed revenue distribution for independent musicians",
    "Water scarcity in the Colorado River Basin and interstate allocation disputes",
    "The rise and business model of the private space launch industry",
    "Sociolinguistic research on language revitalization in Indigenous communities",
]

In [26]:
JUDGE_MODEL = "gpt-4.1"

### Evals run 1

In [33]:
query = "OpenAI"
print("Testing:", query)

try:
    results = DDGS().text(query, max_results=3)
    print(results)
except Exception as error:
    print(type(error).__name__, repr(error))

Testing: OpenAI
[{'title': 'OpenAI', 'href': 'https://en.wikipedia.org/wiki/OpenAI', 'body': "OpenAI is an American artificial intelligence (AI) public benefit corporation headquartered in San Francisco. It develops proprietary generative AI models, particularly its GPT series of large language models (LLMs). Its release of ChatGPT in November 2022 has been credited with catalyzing the AI boom; as of 2026 ChatGPT is the fifth-most-visited website globally. OpenAI also releases Codex, an AI coding agent, and the GPT Image models. In April 2026, OpenAI closed a funding round with a post-money valuation of $852 billion.OpenAI was founded in 2015 in Delaware as a nonprofit, with Sam Altman and Elon Musk as co-chairs; Musk resigned in 2018. A for-profit subsidiary was created in 2019, and a 2025 restructuring converted the subsidiary into OpenAI Group PBC, that is 26% owned by the nonprofit OpenAI Foundation. Microsoft has invested over US$13 billion into OpenAI. OpenAI partners with semico

In [34]:
brief = run_research_agent(TOPICS[0])


🔍 Starting research on: The current state of solid-state battery commercialization

--- Iteration 2 ---
 🔧 Calling function search_web with argument: {'query': 'current state of solid-state battery commercialization 2024'}
Query: 'current state of solid-state battery commercialization 2024'
Results: 3

--- Iteration 3 ---
 🔧 Calling function fetch_url with argument: {'url': 'https://www.startus-insights.com/innovators-guide/solid-state-battery-news-brief/'}
 ❌ Failed to fetch or extract text.
 🔧 Calling function fetch_url with argument: {'url': 'https://pubs.rsc.org/qm/article/8/8/1861/822111/Recent-advances-in-all-solid-state-batteries-for'}
 ❌ Failed to fetch or extract text.
 🔧 Calling function fetch_url with argument: {'url': 'https://www.cas.org/resources/cas-insights/solid-state-battery-technology'}
 ✅ Got text: 24615 chars

--- Iteration 4 ---
 🔧 Calling function search_web with argument: {'query': 'solid-state battery commercialization companies 2024'}
Query: 'solid-state batt

In [38]:
def search_web(query: str):
    print("Query:", repr(query))
    try:
        results = DDGS().text(query, max_results=3)
        print("Results:", len(results))
        return json.dumps(results, indent=2)
    except Exception as error:
        print(type(error).__name__, repr(error))
        return f"No search results available for query: {query}"

In [39]:
import contextlib
import io

# --- Run evals ---
results = []

for topic in TOPICS:
    with contextlib.redirect_stdout(io.StringIO()):
        brief = run_research_agent(topic)
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        max_tokens=5,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"<research_brief>\n{brief}\n</research_brief>"},
        ],
    )
    verdict = response.choices[0].message.content.strip().upper()
    results.append((topic, verdict, brief))
    print(f"{verdict:6} | {len(brief):6,} chars | {topic}")

failures = sum(1 for _, v, _ in results if v == "TRUE")
print(f"\nFailure rate: {failures}/{len(results)}")

TRUE   |  5,767 chars | The current state of solid-state battery commercialization
TRUE   |  3,868 chars | How GLP-1 receptor agonists are reshaping obesity treatment guidelines
TRUE   |  7,079 chars | Economic effects of remote work on mid-sized U.S. downtowns
TRUE   |  7,127 chars | The 2015 Paris Agreement's Article 6 carbon market rules and their implementation status
FALSE  |  7,844 chars | Recent advances in room-temperature superconductor research and replication attempts
TRUE   |  4,253 chars | The regulatory landscape for autonomous trucking in the European Union
TRUE   |  6,327 chars | How streaming economics changed revenue distribution for independent musicians
FALSE  |  6,890 chars | Water scarcity in the Colorado River Basin and interstate allocation disputes
FALSE  |  7,521 chars | The rise and business model of the private space launch industry
TRUE   |  6,694 chars | Sociolinguistic research on language revitalization in Indigenous communities

Failure rate: 7/10


### Evals run 2

In [40]:
RESEARCH_AGENT_PROMPT = """
You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. Try to gather information from at least 5-6 different sources
7. When you have enough information, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
It CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

In [42]:
import contextlib
import io

# --- Run evals ---
results = []

for topic in TOPICS:
    print("\n","="*60,"\n\n")
    with contextlib.redirect_stdout(io.StringIO()):
        brief = run_research_agent(topic)
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        max_tokens=5,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"<research_brief>\n{brief}\n</research_brief>"},
        ],
    )
    verdict = response.choices[0].message.content.strip().upper()
    results.append((topic, verdict, brief))
    print(f"{verdict:6} | {len(brief):6,} chars | {topic}")
    if verdict == "FALSE":
        print(brief)

failures = sum(1 for _, v, _ in results if v == "TRUE")
print(f"\nFailure rate: {failures}/{len(results)}")




TRUE   |  5,673 chars | The current state of solid-state battery commercialization



TRUE   |  5,616 chars | How GLP-1 receptor agonists are reshaping obesity treatment guidelines



TRUE   |  7,477 chars | Economic effects of remote work on mid-sized U.S. downtowns



TRUE   |  7,449 chars | The 2015 Paris Agreement's Article 6 carbon market rules and their implementation status



TRUE   |  5,556 chars | Recent advances in room-temperature superconductor research and replication attempts



TRUE   |  5,889 chars | The regulatory landscape for autonomous trucking in the European Union



TRUE   |  6,204 chars | How streaming economics changed revenue distribution for independent musicians



TRUE   |  4,947 chars | Water scarcity in the Colorado River Basin and interstate allocation disputes



TRUE   |  7,376 chars | The rise and business model of the private space launch industry



TRUE   |  5,151 chars | Sociolinguistic research on language revitalization in Indigenous communi

### Evals best practices

1. Code-based where possible
2. Just Output vs Traces
3. Involve the product person
4. One judge per failure mode
5. Binary score is better
6. Run evals across diverse inputs